In [11]:
import pandas
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from peft import LoraConfig, get_peft_model

dataset_path = r"../../datasets/sentiment_dataset.csv"

In [12]:
dataset = pandas.read_csv(dataset_path)

In [13]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["label"])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [14]:
checkpoint = "vinai/phobert-base" 

In [15]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# PhoBERT (RoBERTa-style) requires fixed max_length to avoid shape mismatches.
max_length = 256

def tokenizer_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_token_type_ids=False,
    )


def preprocessing(ds):
    ds = ds.map(tokenizer_fn, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

In [17]:
print(train_dataset['text'][0])

 Đến lần muốn đến thêm lần ngon tuyệt


In [18]:
tokenized_train = preprocessing(train_dataset)
tokenized_test = preprocessing(test_dataset)

Map: 100%|██████████| 882/882 [00:00<00:00, 2204.80 examples/s]


In [19]:
# target_modules = ["q_proj", "v_proj"]

## WARNING: Bert không dùng q_proj,v_proj, 
# Sử dụng ["query", "value"] 
# https://medium.com/@simon.gsponer/a-comprehensive-guide-ii-finetuning-a-bert-llm-with-lora-and-make-it-pipeline-compatible-9508e3822907

target_modules = ["query", "value"]

In [20]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,  
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,181,954 || all params: 136,181,764 || trainable%: 0.8679


In [21]:
from transformers import Trainer, TrainingArguments

In [22]:
training_args = TrainingArguments(
    output_dir="../../models/lora_phobert_lora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",   # Tránh bị hỏi login wandb trong Colab
    disable_tqdm=False,
)


In [23]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer)

In [24]:
# Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [25]:
train_result = trainer.train()
print(f"Final training loss: {train_result.training_loss:.4f}")


Epoch,Training Loss,Validation Loss
1,0.211700,0.211943
2,0.264000,0.187010
3,0.185200,0.186515


Final training loss: 0.2351


In [26]:
model_output_dir = "../../output/phobert_lora_sentiment"
model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)


('../../output/phobert_lora_sentiment\\tokenizer_config.json',
 '../../output/phobert_lora_sentiment\\special_tokens_map.json',
 '../../output/phobert_lora_sentiment\\vocab.txt',
 '../../output/phobert_lora_sentiment\\bpe.codes',
 '../../output/phobert_lora_sentiment\\added_tokens.json')